# 01a — Scrape ahkam.sjc.bh (Court of Cassation)

**Status: search + fetch pipeline confirmed working live.** Validated via live browser testing:
- Plain `requests`/`httpx` GETs to `Htm/File.aspx` work fine even with 0 cookies — the fingerprint
  check does not actually gate that endpoint. `aSearch`, however, needs the correct payload (below).
- `POST /Default.aspx/aSearch` — real payload schema confirmed by reading `/Scripts/TJquery.js`'s
  `getR()` function directly: `{Y1,Y2,J1,J2,T,C,S,P,L,A,N,G}`, where `T` = type code and `G` = page
  number (1-indexed, string). Standard double-quoted JSON works fine — quote style doesn't matter,
  only field names do. Confirmed live: page 1 vs page 2 for type `M` return different result sets.
- **All 6 type codes confirmed** by reading the search form's `grpType` radio inputs directly:
  مدني=`M`, جنائي=`J`, شرعي=`S`, تجاري=`T`, انتخابات=`E`, توحيد المبادئ=`P`.
- `GET /Htm/File.aspx?i=803%20M%202025%20K%200` — returns the FULL judgment as clean HTML
  (Word-generated markup, Arabic as numeric HTML entities — must `html.unescape()` before checking
  for Arabic content). No PDF, no image, no OCR.
- Corpus size by type (~25 records/page, unconfirmed exact totals): مدني 257 pages (~6,425), جنائي 92
  (~2,300), شرعي 11 (~275), تجاري 7 (~175), انتخابات 1 (15 exact), توحيد المبادئ 2 (~50).
  **Total ≈ 9,240 records.**
- **Pagination has isolated mid-sequence gaps — confirmed live.** Type `M` (مدني) page 22 returned
  `"NO"` while pages 21 and 23 both had real results, and page 257 was the true (clean) end. A first
  full-pull attempt that stopped on the first empty page cut مدني off at 525/~6,425 and شرعي at
  125/~275. Fixed by requiring several **consecutive** empty pages before concluding a type is done,
  instead of stopping on the first one.
- `pull_all` is resumable — re-running it skips judgments already saved to disk from an earlier run,
  only fetching new ones. Useful since most of this corpus doesn't change day to day.

**Strategy:** open one Selenium session (kept mainly for a realistic User-Agent/Referer; the search+fetch
endpoints themselves did not require cookies in live testing), then replicate `aSearch` +
`Htm/File.aspx` calls directly via `requests` instead of clicking through thousands of rows in the
browser.


In [1]:
pip install selenium

Note: you may need to restart the kernel to use updated packages.


In [2]:
import time
import re
import json
from pathlib import Path

import requests
from selenium import webdriver
from selenium.webdriver.common.by import By

BASE = "https://ahkam.sjc.bh"
OUT_DIR = Path("../data/raw/sjc")
OUT_DIR.mkdir(parents=True, exist_ok=True)


### Step 1 — open a real browser session to pass the fingerprint check, then extract cookies

In [3]:
driver = webdriver.Chrome()
driver.get(BASE + "/")
time.sleep(2)  # let the page + any JS challenge settle

cookies = driver.get_cookies()
session = requests.Session()
for c in cookies:
    session.cookies.set(c["name"], c["value"])
session.headers.update({
    "User-Agent": driver.execute_script("return navigator.userAgent"),
    "Referer": BASE + "/",
})

print(f"Captured {len(cookies)} cookies from the browser session")

Captured 0 cookies from the browser session


### Step 2 — type codes (confirmed, no longer a TODO)

Confirmed live by reading the search form's `input[name="grpType"]` radio values directly via
browser devtools — no guessing involved. See `TYPE_CODES` below.

In [4]:
# Confirmed live via browser devtools (input[name="grpType"] radio values on the search form) —
# no longer guessed.
TYPE_CODES = {
    "مدني": "M",
    "جنائي": "J",
    "شرعي": "S",
    "تجاري": "T",
    "انتخابات": "E",
    "توحيد المبادئ": "P",
}


### Step 3 — replicate the `aSearch` API to enumerate all records for a given type

In [5]:
def search_page(session, type_code: str, page: int, retries: int = 4, delay: float = 2.0) -> str:
    """Replicates the aSearch PageMethod call. Returns the raw HTML results-table fragment.
    Payload schema confirmed live by reading /Scripts/TJquery.js's getR() function (the real
    handler behind the search form): fields are Y1/Y2 (year from/to), J1/J2 (session date from/to),
    T (type code), C (category codes, comma-joined), S (subcategory codes), P/L/A/N (keyword search
    modes), G (page number, 1-indexed, as a string). At least one field must be non-empty or the
    server returns 'NO'. Quote style (single vs double) doesn't matter — only field names do.
    Retries with backoff — lloc.gov.bh showed this class of site returns transient errors under
    sustained load, so a long multi-hour pull needs this to not die on a single blip.
    """
    url = f"{BASE}/Default.aspx/aSearch"
    payload = {
        "Y1": "", "Y2": "", "J1": "", "J2": "",
        "T": type_code,
        "C": "", "S": "", "P": "", "L": "", "A": "", "N": "",
        "G": str(page),
    }
    last_exc = None
    for attempt in range(retries):
        try:
            r = session.post(url, json=payload, headers={"Content-Type": "application/json; charset=UTF-8"})
            r.raise_for_status()
            data = r.json()
            d = data.get("d", "")
            if d in ("NO", "NIL"):
                return ""
            return d
        except Exception as e:
            last_exc = e
            time.sleep(delay * (attempt + 1))
    raise last_exc


def extract_keys(html_fragment: str):
    """Pulls all showModal('...') keys out of a results-table HTML fragment."""
    return re.findall(r"showModal\('([^']+)'\)", html_fragment)


### Step 4 — fetch full judgment text for a given key

In [6]:
def fetch_judgment(session, key: str, retries: int = 4, delay: float = 2.0) -> str:
    """key example: '803 M 2025 K 0' -> full judgment HTML (Word-generated markup).
    Retries with backoff for the same reason as search_page — long unattended runs need this.
    """
    from urllib.parse import quote
    url = f"{BASE}/Htm/File.aspx?i={quote(key)}"
    last_exc = None
    for attempt in range(retries):
        try:
            r = session.get(url)
            r.raise_for_status()
            return r.text
        except Exception as e:
            last_exc = e
            time.sleep(delay * (attempt + 1))
    raise last_exc


# Smoke test on the one key we validated live this session
sample = fetch_judgment(session, "803 M 2025 K 0")
print(sample[:500])


<html>

<head>
<meta http-equiv=Content-Type content="text/html; charset=windows-1252">
<meta name=Generator content="Microsoft Word 15 (filtered)">
<style>
<!--
 /* Font Definitions */
 @font-face
	{font-family:SimSun;
	panose-1:2 1 6 0 3 1 1 1 1 1;}
@font-face
	{font-family:"Cambria Math";
	panose-1:2 4 5 3 5 4 6 3 2 4;}
@font-face
	{font-family:Calibri;
	panose-1:2 15 5 2 2 2 4 3 2 4;}
@font-face
	{font-family:Cambria;
	panose-1:2 4 5 3 5 4 6 3 2 4;}
@font-face
	{font-fam


### Step 5 — full pull (pacing required)

Per the site's own behaviour observed this session for `lloc.gov.bh`, and as general good practice —
pace requests (~1s+ gaps), and wrap each call in retry logic. `sjc.bh` was not stress-tested for burst
sensitivity the way `lloc.gov.bh` was, so treat conservative pacing as the safe default here too.

In [7]:
def pull_all(session, type_name: str, type_code: str, out_dir: Path, delay: float = 1.0,
             consecutive_empty_to_stop: int = 5):
    if type_code is None:
        raise ValueError(f"No confirmed type code for {type_name} — resolve the TODO above first")

    # NOTE: the site has isolated empty pages mid-sequence (confirmed live: type M page 22 returned
    # "NO" while pages 21 and 23 both had real results) — a single empty page does NOT mean the
    # results have ended. Stopping on the first empty page cut مدني off at 525/~6,425 and شرعي at
    # 125/~275 in an earlier run. Fix: only stop after several CONSECUTIVE empty pages in a row.
    page = 1
    all_keys = []
    consecutive_empty = 0
    max_page_seen_with_results = 0
    while consecutive_empty < consecutive_empty_to_stop:
        html = search_page(session, type_code, page)
        keys = extract_keys(html)
        if keys:
            all_keys.extend(keys)
            consecutive_empty = 0
            max_page_seen_with_results = page
            print(f"{type_name} page {page}: {len(keys)} records (running total {len(all_keys)})")
        else:
            consecutive_empty += 1
            print(f"{type_name} page {page}: empty ({consecutive_empty}/{consecutive_empty_to_stop} consecutive)")
        page += 1
        time.sleep(delay)

    print(f"{type_name}: stopped after {consecutive_empty_to_stop} consecutive empty pages "
          f"(last page with results: {max_page_seen_with_results})")

    # Resumable: if this type's file already exists from an earlier run (e.g. جنائي/تجاري/انتخابات/
    # توحيد_المبادئ already look complete), skip keys already saved instead of re-fetching everything.
    out_path = out_dir / f"sjc_{type_name}.json"
    records = []
    already_saved = {}
    if out_path.exists():
        already_saved = {r["key"]: r for r in json.loads(out_path.read_text(encoding="utf-8"))}
        print(f"{type_name}: {len(already_saved)} judgments already saved from a previous run — will skip those")

    new_keys = [k for k in all_keys if k not in already_saved]
    records.extend(already_saved.get(k) for k in all_keys if k in already_saved)
    print(f"{type_name}: {len(new_keys)} new judgments to fetch")

    for key in new_keys:
        try:
            html = fetch_judgment(session, key)
            records.append({"key": key, "html": html})
        except Exception as e:
            print(f"FAILED key={key}: {e}")
        time.sleep(delay)

    out_path.write_text(json.dumps(records, ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"Saved {len(records)} judgments -> {out_path}")
    return records


# Full pull across all 6 case types. Resumable (see pull_all) — types that already look complete
# from the previous run (جنائي/تجاري/انتخابات/توحيد_المبادئ) will only re-check pagination, not
# re-fetch judgments already saved. مدني and شرعي need the bulk of the remaining work since the
# pagination bug cut them short before. Each type is wrapped in its own try/except so a persistent
# failure in one type doesn't lose progress already made on the others.
all_records = {}
for type_name, type_code in TYPE_CODES.items():
    key_safe = type_name.replace(" ", "_")
    try:
        all_records[type_name] = pull_all(session, key_safe, type_code, OUT_DIR)
        print(f"=== {type_name} done: {len(all_records[type_name])} judgments ===\n")
    except Exception as e:
        print(f"=== {type_name} FAILED entirely: {e} ===\n")
        all_records[type_name] = []

print(f"\nGRAND TOTAL: {sum(len(v) for v in all_records.values())} judgments across all 6 types")


مدني page 1: 25 records (running total 25)


مدني page 2: 25 records (running total 50)


مدني page 3: 25 records (running total 75)


مدني page 4: 25 records (running total 100)


مدني page 5: 25 records (running total 125)


...

توحيد_المبادئ page 4: empty (2/5 consecutive)


توحيد_المبادئ page 5: empty (3/5 consecutive)


توحيد_المبادئ page 6: empty (4/5 consecutive)


توحيد_المبادئ page 7: empty (5/5 consecutive)


توحيد_المبادئ: stopped after 5 consecutive empty pages (last page with results: 2)
توحيد_المبادئ: 27 judgments already saved from a previous run — will skip those
توحيد_المبادئ: 0 new judgments to fetch
Saved 27 judgments -> ..\data\raw\sjc\sjc_توحيد_المبادئ.json
=== توحيد المبادئ done: 27 judgments ===


GRAND TOTAL: 9094 judgments across all 6 types


In [8]:
def verify_scrape(key: str = "803 M 2025 K 0"):
    import re
    html = fetch_judgment(session, key)

    arabic_chars = len(re.findall(r"[\u0600-\u06FF]", html))
    has_arabic = arabic_chars > 200  # a real judgment has thousands, an error page has ~0

    print(f"Response length: {len(html)} chars")
    print(f"Arabic characters found: {arabic_chars}")
    print(f"Looks like real judgment text: {'✅ YES' if has_arabic else '❌ NO — check session/cookies'}")
    print("\n--- First 300 chars ---")
    print(html[:300])

verify_scrape()

Response length: 49801 chars
Arabic characters found: 0
Looks like real judgment text: ❌ NO — check session/cookies

--- First 300 chars ---
<html>

<head>
<meta http-equiv=Content-Type content="text/html; charset=windows-1252">
<meta name=Generator content="Microsoft Word 15 (filtered)">
<style>
<!--
 /* Font Definitions */
 @font-face
	{font-family:SimSun;
	panose-1:2 1 6 0 3 1 1 1 1 1;}
@font-face
	{font-family:"Cambria Ma


In [9]:
def verify_scrape(key: str = "803 M 2025 K 0"):
    import re
    import html as html_module  # stdlib, decodes &#1580; etc.

    raw_html = fetch_judgment(session, key)
    decoded = html_module.unescape(raw_html)

    arabic_chars = len(re.findall(r"[\u0600-\u06FF]", decoded))
    has_arabic = arabic_chars > 200

    print(f"Response length: {len(raw_html)} chars")
    print(f"Arabic characters after decoding entities: {arabic_chars}")
    print(f"Looks like real judgment text: {'✅ YES' if has_arabic else '❌ NO — check session/cookies'}")
    print("\n--- First 300 chars of decoded Arabic text ---")
    # strip tags crudely just for a readable preview
    text_only = re.sub(r"<[^>]+>", " ", decoded)
    text_only = re.sub(r"\s+", " ", text_only).strip()
    print(text_only[:300])

verify_scrape()

Response length: 49801 chars
Arabic characters after decoding entities: 5471
Looks like real judgment text: ✅ YES

--- First 300 chars of decoded Arabic text ---
جلسة 22 من يوليو سنة 20 25 برئاسة المستشار الدكتور/ طه عبدالمولى طه الوكيل بالمحكمه وعدنان عبدالله الشيخ هزيم الشامسي وإبــراهـــيــم مــحـمــد الـمرصفاوي الوكيلان بالمحكمه وعبدالعزيز عبدالعزيز أحمد فرحات ( ) الطعن رقم 1/00803/2024/10 دعوى" مصاريف الدعوى: أـتعاب المحاماه". رسوم. 1 . إلزام الخصم الذي
